# Bloom — Upload recovery
Use this when a completed continuation has been saved to Drive but upload failed or the runtime ended. No data processing or retraining. Requires completed metrics, evaluation, provenance, tokenizer, config and weights; incomplete runs are rejected. Only run with permission to publish.

In [ ]:
# 1. Restore code and lightweight HF client; no GPU or training required
import os,sys,subprocess,json
from pathlib import Path
IN_COLAB='google.colab' in sys.modules
IN_KAGGLE=Path('/kaggle/working').is_dir()
REPO=Path('/content/centauri-bloom') if IN_COLAB else Path.cwd()
if not (REPO/'bloom'/'notebook_workflow.py').exists():
    if (REPO/'.git').exists():
        subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
    elif IN_COLAB:
        subprocess.run(['git','clone','--depth','1','https://github.com/InterstellarNerd8760/centauri-bloom.git',str(REPO)],check=True)
    else:
        raise RuntimeError('Open this notebook from the updated Bloom repository.')
sys.path.insert(0,str(REPO))
os.environ['HF_HUB_DISABLE_XET']='1'
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub','ipywidgets'],check=True)
from bloom.notebook_workflow import submit_candidate


In [ ]:
# 2. Select completed run directory
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
RUN_DIR=''  # Paste the completed run folder here
if not RUN_DIR:
    raise ValueError('Enter the exact completed run folder.')
OUT_DIR=Path(RUN_DIR).expanduser()
metrics=json.loads((OUT_DIR/'metrics.json').read_text())
RUN_NAME=metrics['run_name']
PARENT_REVISION=metrics['parent_revision']
STEPS=metrics['additional_steps']
HF_REPO_ID='Centauri-Bloom/Bloom-Text-0.1'
SUBMIT_HF_PR=True
print('Candidate:',RUN_NAME,'| steps:',STEPS,'| parent:',PARENT_REVISION)


In [ ]:
# 3. Authorize candidate submission
from huggingface_hub import HfApi
from bloom.notebook_workflow import auth_client
AUTH_API = None
if not SUBMIT_HF_PR:
    AUTH_API = HfApi()
    print('Local-only run: no upload.')
elif IN_COLAB:
    from IPython.display import display
    import ipywidgets as widgets
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
    token_field=widgets.Password(description='HF token:',placeholder='Paste token')
    authorize=widgets.Button(description='Authorize upload',button_style='success')
    auth_out=widgets.Output()
    def authenticate(_):
        global AUTH_API
        with auth_out:
            auth_out.clear_output()
            try:
                AUTH_API=auth_client(HF_REPO_ID,token_field.value)
                token_field.value=''
                authorize.disabled=True
                print('Authenticated. Candidate PR only. HF checks write permissions on submission.')
            except Exception as exc:
                print('Authentication failed:',type(exc).__name__)
    authorize.on_click(authenticate)
    display(token_field,authorize,auth_out)
else:
    from getpass import getpass
    token=os.environ.get('HF_TOKEN')
    if not token and IN_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            token=UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            pass
    AUTH_API=auth_client(HF_REPO_ID,token or getpass('HF token: '))
    del token


In [ ]:
# 8. Submit the completed candidate. Rerun only this cell for upload retries.
if SUBMIT_HF_PR:
    if AUTH_API is None:
        raise RuntimeError('Authorize upload first.')
    submit_candidate(AUTH_API,HF_REPO_ID,OUT_DIR,STEPS,PARENT_REVISION,RUN_NAME)
else:
    print('Local candidate saved:',OUT_DIR)
